In [15]:
%run aaa_setup.ipynb

In [16]:
class Filter(SetUp): # type: ignore

    def __init__(self):
        super().__init__()
        return
    
    def filter(self, cutoff_institutions=None, cutoff_authors=None):
        self._filter_institutions(cutoff=cutoff_institutions)
        self._filter_authors(cutoff=cutoff_authors)
        self._modify_authorships()
        self._modify_works()
        self._modify_cites()
        return self
    
    def _filter_institutions(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.institutions AS        
                WITH get_work_institution_CTE AS (
                    SELECT DISTINCT work_id, institution_id 
                    FROM project.authorships_full
                ),
                institution_counts AS (
                    SELECT
                        institution_id,
                        COUNT(work_id) AS works_count
                    FROM get_work_institution_CTE
                    GROUP BY institution_id
                ),
                ranked_institutions AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count DESC) AS row_number,
                        institution_id,
                        works_count
                    FROM institution_counts
                )

                SELECT *
                    FROM ranked_institutions
                    WHERE works_count >= {cutoff}
                    ORDER BY works_count DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_institutions FROM memory.institutions").show()
        self.db.sql("SELECT count(DISTINCT institution_id) AS count_institutions_original FROM project.authorships_full").show()
        return self

    def _filter_authors(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.authors AS
                WITH get_work_author_CTE AS (
                    SELECT DISTINCT work_id, author_id, list_sort(list(publication_year))[1] AS first_year
                    FROM project.authorships_full
                    LEFT JOIN project.works_full
                    USING (work_id)
                    GROUP BY work_id, author_id
                ),
                author_counts_CTE AS (
                    SELECT
                        author_id,
                        COUNT(work_id)/(2026-first_year) AS works_count_prorata
                    FROM get_work_author_CTE
                    GROUP BY author_id, first_year
                ),
                ranked_authors_CTE AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count_prorata DESC) AS row_number,
                        author_id,
                        works_count_prorata
                    FROM author_counts_CTE
                )

                SELECT *
                FROM ranked_authors_CTE
                WHERE works_count_prorata >= {cutoff}
                ORDER BY works_count_prorata DESC
        """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_authors FROM memory.authors").show()
        self.db.sql("SELECT count(DISTINCT author_id) AS count_authors_original FROM project.authorships_full").show()
        return self
    
    def _modify_authorships(self):
        sql = """
            CREATE OR REPLACE TABLE memory.authorships AS
            WITH
            filtered_works_authors_CTE AS
                (SELECT a.* 
                    FROM project.authorships_full a
                    WHERE author_id IN (SELECT author_id FROM memory.authors))

            SELECT a.* 
                FROM filtered_works_authors_CTE a
                WHERE institution_id IN (SELECT institution_id FROM memory.institutions)
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_authorships FROM memory.authorships").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_authorships_original FROM project.authorships").show()      

    def _modify_works(self):
        sql = """
            CREATE OR REPLACE TABLE memory.works AS
            SELECT DISTINCT w.* 
                FROM project.works_full w
                WHERE work_id in (SELECT work_id FROM memory.authorships)
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_works FROM memory.works").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_works_original FROM project.works").show()
        return self

  

    def _modify_cites(self):
        sql = """
            CREATE OR REPLACE TABLE memory.cited AS
            WITH
            citer_cited_CTE AS
                (SELECT work_id, 
                        unnest(referenced_works) AS cited_id
                    FROM project.cited_full
                ),
            filter_citer_cited_CTE AS
                (SELECT work_id,
                        cited_id
                    FROM citer_cited_CTE
                    WHERE cited_id IN (SELECT work_id FROM memory.works))

            SELECT work_id, 
                    list(cited_id)
                FROM filter_citer_cited_CTE
                WHERE work_id in (SELECT work_id FROM memory.works)
                GROUP BY work_id
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_citers FROM memory.cited").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_citers_original FROM project.cited").show() 
        return

#### This cell matches Domingo's C and T lists to authors in the OpenAlex extract from the Journal Set  

- Extract Domingo's list and ensure that the names are normalised

- Compare with OpenAlex lists  

    - HCRs - endogenous - from OpenAlex references in journal set  
    - Authorships - endogenous - from OpenAlex works in journal set   
    - Authors - exogenous - from the entire OpenAlex author dataest, filtered into eeconomics and Business topics    


In [ ]:
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        self.db.sql("CREATE OR REPLACE TABLE project.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM project.domingo_sample").df()
        return
    
    def match_sample(self):
        print('_match_sample')
        sample = self.db.sql("SELECT * FROM project.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        authors = self.db.sql("SELECT * FROM project.authors").df().sort_values(['cited_by_count', 'works_count'], ascending=[False, False])\
            .reset_index()[['author_id', 'author_name', 'orcid', 'display_name_alternatives', 'cited_by_count', 'works_count']]
        # authors = authors[authors.cited_by_count>100].dropna()
        for row in authors.itertuples():
            authors.at[row.Index, 'display_name_alternatives'] = [normalise_name(n)[-1] if isinstance(n, str) else n for n in row.display_name_alternatives]
        print(f'{authors.shape = }\n{authors.head()}')

        for row in sample.itertuples():
            for row1 in authors.itertuples():
                if row.fullname in row1.display_name_alternatives:
                    sample.at[row.Index, 'author_id'] = row1.author_id
                    sample.at[row.Index, 'orcid'] = row1.orcid
                    break
        print(f'{sample.shape = }\n{sample.head(24)}')
        print(f'{sample[sample.author_id.isna()].shape = }\n{sample[sample.author_id.isna()].head(24)}')
        self.db.sql("CREATE OR REPLACE TABLE memory.matched AS SELECT * FROM sample")
        return

    def load_sample(self):
        print('compare sample')
        sql = """
            CREATE OR REPLACE TABLE project.sample_matched AS
                SELECT Research_Profile,
                        m.orcid,
                        m.author_id,
                        m.first,
                        m.last,
                        m.fullname,
                        ACR,
                        PUB,
                        CIT,
                        HCP,
                        suma,          
                        coc,      
                        score, 
                        "Group",
                        works_count_endogenous, 
                        citations_endogenous,
                        hca_endogenous,	                    	
                        works_count_total,
                        cited_by_count,
                        hca_total,
                        "2yr_mean_citedness",
                        h_index,
                        citations_total_ AS citations_total_oa,	
                    FROM memory.matched m 
                    LEFT JOIN citation_summary s
                        ON s.author_id = m.author_id
                    ORDER BY hca_endogenous DESC, citations_endogenous DESC
            """
        self.db.sql(sql)
        sample_align = self.db.sql("SELECT * FROM project.sample_matched").df()
        print(f'{sample_align.shape = }\n{sample_align.head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample_align.to_excel(writer, index=False, sheet_name='full_match')
            df = sample_align.groupby('Research_Profile').first().reset_index()
            df.to_excel(writer, index=False, sheet_name='filtered')
            print(f'{df.shape = }\n{df.head()}')
            df[df.author_id.isna()].to_excel(writer, index=False, sheet_name='filtered_unmatched')
            print(f'{df[df.author_id.isna()].shape = }\n{df[df.author_id.isna()].head()}')
        return

In [18]:
def main():

    f = Filter()
    f.filter(cutoff_institutions=150, cutoff_authors=2)

    mds = MatchDomingoSample()
    mds.extract_sample()
    mds.match_sample()
    mds.load_sample()

    return

In [19]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ memory   │ main    │ t1                   │ [i, j]               │ [INTEGER, INTEGER]                    │ false     │
│ project  │ main    │ author_works_counts  │ [author_id, author…  │ [VARCHAR, VARCHAR, BIGINT, BIGINT, …  │ false     │
│ project  │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorshi

CatalogException: Catalog Error: Table with name citation_summary does not exist!
Did you mean "project.citation_summary"?

LINE 27:                     LEFT JOIN citation_summary s
                                       ^